# Experimento 1 — Cobertura de especies amenazadas en BirdCLEF+ 2025

**Pregunta:** ¿el entrenamiento disponible permite incluir al paujil de pico azul (*Crax alberti*) y al chavarrí (*Chauna chavarria*, escrito *Chauna chavaria* en la taxonomía del dataset)? ¿Cómo se comparan con las 16 especies de Hito 1?

Este análisis es adicional y no modifica el notebook ni los resultados del Hito 1. Inspecciona etiquetas y metadatos de entrenamiento, mide cobertura geográfica en un radio de 300 km de la Reserva El Silencio, contrasta volumen/calidad con las especies seleccionadas y aplica los umbrales de elegibilidad documentados en el preprocesamiento. **Conclusión para el ejercicio actual:** no conviene sumar estas dos especies a las 16 clases; después de los filtros de Hito 1, su inclusión llevaría la razón de desbalance máximo/mínimo de `6.77:1` a `70.14:1`. Esto indica insuficiencia y desbalance de datos, no que las especies sean imposibles de identificar. El experimento no entrena un clasificador ni mide capacidad predictiva.

**Fuente de conservación:** [Fundación Biodiversa Colombia — Reserva El Silencio](https://www.fundacionbiodiversa.org/fundacion2024/es/el-silencio/), que resalta la presencia de ambas especies entre la fauna amenazada de la reserva. El texto usa *Chauna chavarria*; para unir con los registros se usa el nombre científico de `taxonomy.csv`, *Chauna chavaria*, y su etiqueta BirdCLEF `norscr1`.

**Kernel:** Python (Proyecto 1 MIAA).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "src" / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "src" / "data" / "processed"
if not (RAW_DIR / "train.csv").is_file():
    raise FileNotFoundError(f"No se encontró el BirdCLEF+ 2025 local en {RAW_DIR}")

train = pd.read_csv(RAW_DIR / "train.csv")
taxonomy = pd.read_csv(RAW_DIR / "taxonomy.csv")
durations = pd.read_csv(PROCESSED_DIR / "train_durations.csv", usecols=["filename", "duration_s"])
data = train.merge(
    taxonomy[["primary_label", "class_name"]], on="primary_label", validate="m:1"
)
data = data.merge(durations, on="filename", validate="1:1")
print(f"Entrenamiento: {len(data):,} grabaciones; {data.primary_label.nunique()} etiquetas")
print(f"Filas con metadatos y duración unidos: {data.duration_s.notna().sum():,}")

Entrenamiento: 28,564 grabaciones; 206 etiquetas
Filas con metadatos y duración unidos: 28,564


In [2]:
# Las 16 etiquetas elegidas en Hito 1 se leen del artefacto existente; no se reescribe.
hito1_species = pd.read_csv(PROCESSED_DIR / "subset_species.csv")
selected_labels = set(hito1_species["primary_label"])

target_names = {"Crax alberti", "Chauna chavaria"}
target_taxonomy = taxonomy[taxonomy.scientific_name.isin(target_names)].copy()
assert set(target_taxonomy.scientific_name) == target_names, (
    "No se encontraron ambas especies en taxonomy.csv; revise el nombre taxonómico."
)
target_labels = set(target_taxonomy.primary_label)
print("Etiquetas objetivo según taxonomy.csv:")
print(target_taxonomy[["primary_label", "scientific_name", "common_name", "class_name"]].to_string(index=False))

# Distancia geodésica aproximada a El Silencio (Yondó), igual que el umbral regional de Hito 1.
lat0, lon0 = np.radians(6.76), np.radians(-74.21)
lat, lon = np.radians(data.latitude), np.radians(data.longitude)
dlat, dlon = lat - lat0, lon - lon0
a = np.sin(dlat / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin(dlon / 2) ** 2
data["dist_km"] = 2 * 6371 * np.arcsin(np.sqrt(a))
data["regional_300km"] = data.dist_km <= 300
data["colombia_bbox"] = (
    data.latitude.between(-4.3, 12.5) & data.longitude.between(-79.1, -66.8)
)

summary = data.groupby("primary_label").agg(
    scientific_name=("scientific_name", "first"),
    common_name=("common_name", "first"),
    n_recordings=("filename", "size"),
    hours=("duration_s", lambda s: s.sum() / 3600),
    n_authors=("author", "nunique"),
    n_rating_ge4=("rating", lambda s: int((s >= 4).sum())),
    n_regional=("regional_300km", "sum"),
    n_colombia=("colombia_bbox", "sum"),
    n_coordinates=("latitude", "count"),
)
# Conteo rating>=4 en el subconjunto regional; se calcula explícitamente por etiqueta.
regional_high_quality = data[data.regional_300km & data.rating.ge(4)].groupby("primary_label").size()
summary["n_regional_rating_ge4"] = regional_high_quality.reindex(summary.index, fill_value=0)
for label in target_labels:
    mentions = data.secondary_labels.fillna("").str.contains(label, regex=False, case=False).sum()
    print(f"Menciones como etiqueta secundaria de {label}: {mentions}")
summary["selected_hito1"] = summary.index.isin(selected_labels)
summary["target"] = summary.index.isin(target_labels)

# Umbrales de elegibilidad de Hito 1; además de ser aves, exige volumen, rating, autores y evidencia regional.
summary["meets_hito1_eligibility"] = (
    summary.n_recordings.ge(120)
    & summary.n_rating_ge4.ge(60)
    & summary.n_authors.ge(40)
    & summary.n_regional.ge(10)
)

columns = [
    "scientific_name", "common_name", "n_recordings", "hours", "n_authors",
    "n_rating_ge4", "n_regional", "n_regional_rating_ge4", "n_colombia",
    "selected_hito1", "meets_hito1_eligibility",
]
print("\nEspecies objetivo:")
print(summary.loc[summary.target, columns].round(2).to_string())
print("\nComparación con las 16 especies elegidas en Hito 1 (ordenadas por grabaciones):")
print(summary.loc[summary.selected_hito1, columns].sort_values("n_recordings", ascending=False).round(2).to_string())

# Aplicar a las especies objetivo los mismos filtros de grabación usados por Hito 1.
target_recordings = data[data.primary_label.isin(target_labels)].copy()
target_keep = (target_recordings.duration_s >= 1.0) & ~(
    (target_recordings.collection == "XC")
    & target_recordings.rating.between(0.1, 1.99)
)
target_recordings = target_recordings[target_keep].drop_duplicates("url", keep="first")
target_counts = target_recordings.groupby("primary_label").size()
hito1_counts = hito1_species.set_index("primary_label").n_incl
current_imbalance = hito1_counts.max() / hito1_counts.min()
combined_imbalance = hito1_counts.max() / target_counts.min()
print("\nDesbalance después de aplicar los mismos filtros de inclusión:")
print(f"Hito 1 (16 especies): {hito1_counts.max()}/{hito1_counts.min()} = {current_imbalance:.2f}:1")
print(f"Al agregar las dos especies objetivo: {hito1_counts.max()}/{target_counts.min()} = {combined_imbalance:.2f}:1")
print(f"Incremento de la razón máximo/mínimo: {combined_imbalance / current_imbalance:.2f}x")

Etiquetas objetivo según taxonomy.csv:
primary_label scientific_name          common_name class_name
      bubcur1    Crax alberti Blue-billed Curassow       Aves
      norscr1 Chauna chavaria    Northern Screamer       Aves
Menciones como etiqueta secundaria de norscr1: 0
Menciones como etiqueta secundaria de bubcur1: 0

Especies objetivo:
               scientific_name           common_name  n_recordings  hours  n_authors  n_rating_ge4  n_regional  n_regional_rating_ge4  n_colombia  selected_hito1  meets_hito1_eligibility
primary_label                                                                                                                                                                             
bubcur1           Crax alberti  Blue-billed Curassow            20   0.24          7             9           8                      5           8           False                    False
norscr1        Chauna chavaria     Northern Screamer            14   0.21         10            

## Lectura del experimento

- Una etiqueta presente en `train.csv` significa que hay ejemplos supervisados, no que haya suficientes para generalizar.
- Las particiones aleatorias de ventanas derivadas de una misma grabación no aumentan el número de ejemplos biológicamente independientes; en la evaluación se deben agrupar por grabación y, cuando sea posible, por autor/sitio.
- El criterio de Hito 1 (`≥120` grabaciones, `≥60` de rating `≥4`, `≥40` autores y `≥10` regionales) es un filtro de diseño, no un umbral universal de identificabilidad.
- **Decisión para este experimento:** tras aplicar a las especies objetivo los mismos filtros de Hito 1, se conservan 20 grabaciones de *Crax alberti* y 14 de *Chauna chavaria*. Añadirlas a las 16 clases elevaría la razón de desbalance de `6.77:1` a `70.14:1` (más de 10 veces). Por ello, no se recomienda incluirlas en las 16 clases del experimento multiclase actual.
- Esta decisión es metodológica y no reduce su importancia de conservación ni demuestra que sean imposibles de identificar; indica que se necesita más audio etiquetado para evaluarlas de forma fiable.
- Una cantidad pequeña de clips focales fuera de la reserva no valida la detección en soundscapes pasivos de El Silencio. Para medir esa capacidad se requieren audios de campo etiquetados o validación experta.